## Homework copying

To identify questions that were copied from homework assignments, all available homework assignments were saved in a single Word document. The document is treated as one continuous reference text, and each question is compared against this text using fuzzy partial-ratio matching.

The matching procedure was first tested on individual questions for which the classification was known in advance.

In [1]:
import pandas as pd

df = pd.read_excel('databases/chat_database_time.xlsx')

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 46 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   chat_id              990 non-null    int64         
 1   question_id          990 non-null    int64         
 2   question             990 non-null    object        
 3   answer               990 non-null    object        
 4   q_time               990 non-null    datetime64[ns]
 5   a_time               990 non-null    object        
 6   id                   990 non-null    object        
 7   T1                   990 non-null    float64       
 8   T2                   990 non-null    float64       
 9   T3                   990 non-null    float64       
 10  T4                   990 non-null    float64       
 11  T1234                990 non-null    float64       
 12  question_length      990 non-null    int64         
 13  answer_length        990 non-null  

In [3]:
from docx import Document

doc = Document('homeworks/homeworks.docx')

# Extract all homework assignments as a single text string
texts = "\n".join([p.text for p in doc.paragraphs])

print(type(texts))
print(len(texts))
print(texts)

<class 'str'>
79484
"A Z változó eloszlása 6 várható érték¶ és 10 szórású normális eloszlás. Mekkora valószín¶séggel esik Z
értéke az alábbi halmazokba?"
"Mintavételes vizsgálattal akarják megmérni néhány városban a rabszolgatartás visszaállítását támogatók arányát. Az alábbiakban a várható érték és a standard hiba megadását kérjük!
Tegyük fel, hogy E városban 20% a támogatók viszont 80% az ellenz®k aránya. Egy, az E városból vett
 n = 120 f®s mintánál a mintában a támogatók aránya % körül lesz, attól ± kb. %-ra.
 n = 550 f®s mintánál a mintában a támogatók aránya % körül lesz, attól ± kb. %-ra.
 n = 1100 f®s mintánál a mintában a támogatók aránya % körül lesz, attól ± kb. %-ra.
 n = 10000 f®s mintánál a mintában a támogatók aránya % körül lesz, attól ± kb. %-ra.
Tegyük fel, hogy F városban 60% a támogatók viszont csak 40% az ellenz®ké. Egy, az F városból vett
 n = 120 f®s mintánál a mintában a támogatók aránya % körül lesz, attól ± kb. %-ra.
 n = 550 f®s mintánál a mintában a tá

Before applying the procedure to the full dataset, the matching score was tested on individual questions, including both a known homework copy and a question that was not copied from homework.

In [4]:
from thefuzz import fuzz

question = df['question'].iloc[151]

score = fuzz.partial_ratio(
    question.lower(),
    texts.lower()
)

print(score, question, len(question))

77 Mi a Z érték? 13


## Initial homework-copying criterion

Questions shorter than 50 characters were excluded because short questions produced less reliable matches during testing.

As a first criterion, questions of at least 50 characters were classified as homework copies when their fuzzy partial-ratio similarity with the reference document reached at least 90%.

In [5]:
def homework_flag(question):
    if not isinstance(question, str):
        return 0

    question = question.strip()
    length = len(question)

    if length < 50:
        return 0

    score = fuzz.partial_ratio(
        question.lower(),
        texts.lower()
    )

    if score >= 90:
        return 1
    else:
        return 0


df['homework'] = df['question'].apply(homework_flag)

print(df['homework'].value_counts())

print(
    df.loc[df['homework'] == 1, ['question', 'homework']]
)

homework
0    891
1     99
Name: count, dtype: int64
                                              question  homework
40   Tegyük fel, hogy a Bohócpárt támogatottsága az...         1
45   Mi az a vágópont, ami levágja?az 5 szabadságfo...         1
47   Ha 1000 tökéletesen szabályos dobókockát teszt...         1
50   Egy gyártó nyereményjátékot hirdet. A nyeremén...         1
65   Reformpedagógiai módszerrel dolgozó iskolában ...         1
..                                                 ...       ...
882  Az Egyesült Államok-beli Wind Energy Associati...         1
890  2. feladat: Az Egyesült Államok-beli Wind Ener...         1
901  mire kérdez rá, mit kell nézni? Fent a két nem...         1
931  Egy mobilszolgáltató 3 ügyfélköréből vett egys...         1
970  Szia! Az lenne a kérdésem, hogy ez milyen típu...         1

[99 rows x 2 columns]


At the 90% matching threshold, 99 questions were identified as homework copies. All identified questions were manually reviewed and confirmed as genuine matches. No false positive classifications were found.

The 50-character minimum was then tested separately by removing the length requirement while retaining the 90% similarity threshold.

In [6]:
def hw_new(question):
    if not isinstance(question, str):
        return 0

    question = question.strip()

    score = fuzz.partial_ratio(
        question.lower(),
        texts.lower()
    )

    if score >= 90:
        return 1
    else:
        return 0


df['hw1'] = df['question'].apply(hw_new)

print(df['hw1'].value_counts())

difference = df[
    (df['hw1'] == 1) &
    (df['homework'] == 0)
]

print(len(difference))

difference[['question', 'homework', 'hw1']]

hw1
0    876
1    114
Name: count, dtype: int64
15


,question,homework,hw1
80,Mi a szignifikanciaérték?,0,1
81,mi a nullhipotézis?,0,1
152,Mi a szignifikanciaérték?,0,1
161,mi a pontbecslés,0,1
207,mi a szignifikancia érték,0,1
218,Hogy,0,1
227,S jelentése,0,1
228,Szórás számítása,0,1
549,MIt fejez ki a szignifikancia érték?,0,1
692,mit jelent a k,0,1


The additional matches identified after removing the minimum character requirement included questions that could have arisen independently of the homework assignments as well as false positive classifications. The minimum character requirement was therefore retained.

In [7]:
df = df.drop(columns=['hw1'])

## Selection of the final matching threshold

The matching threshold was then relaxed while retaining the 50-character minimum. Thresholds of 85%, 80%, 75%, and 70% were considered.

The differences from the original 90% classification were examined manually to assess whether lowering the threshold improved the detection of genuine homework copies without introducing excessive false positives.

In [8]:
def hw_new2(question):
    if not isinstance(question, str):
        return 0

    question = question.strip()
    length = len(question)

    if length < 50:
        return 0

    score = fuzz.partial_ratio(
        question.lower(),
        texts.lower()
    )

    if score >= 75:
        return 1
    else:
        return 0


df['hw2'] = df['question'].apply(hw_new2)

print(df['hw2'].value_counts())

difference = df[
    (df['hw2'] == 1) &
    (df['homework'] == 0)
]

print(len(difference))

difference[['question', 'homework', 'hw2']]

hw2
0    865
1    125
Name: count, dtype: int64
26


,question,homework,hw2
27,a) Ha a barnapártiak pontosan 15%-nyian lennén...,0,1
31,"Egy párt a legutóbbi választáson 36,1%-ot szer...",0,1
33,"Egy párt a legutóbbi választáson 36,1%-ot szer...",0,1
39,(összesen 25 pont) Egy párt tavaly a választás...,0,1
46,Adjon intervallumbecslést:1500 fős egyszerű vé...,0,1
48,Egy gyártó nyereményjátékot hirdet. A nyeremén...,0,1
266,2. Mintavételes vizsgálattal akarják megmérni ...,0,1
311,"Tegyük fel, hogy korábbi adatok szerint a magy...",0,1
335,És ez esetben mekkora lesz a tesztstatisztika ...,0,1
350,Hogyan interpretálható a regressziós b-együtth...,0,1


A 75% similarity threshold was selected as the final criterion. Lowering the threshold further resulted in an excessive number of false positive classifications.

All questions classified as homework copies under the final criterion were subsequently reviewed manually.

In [9]:
df = df.drop(columns=['homework', 'hw2'])

def homework_flag(question):
    if not isinstance(question, str):
        return 0

    question = question.strip()
    length = len(question)

    if length < 50:
        return 0

    score = fuzz.partial_ratio(
        question.lower(),
        texts.lower()
    )

    if score >= 75:
        return 1
    else:
        return 0


df['homework'] = df['question'].apply(homework_flag)

print(df['homework'].value_counts())

print(
    df.loc[df['homework'] == 1, ['question', 'homework']]
)

homework
0    865
1    125
Name: count, dtype: int64
                                              question  homework
27   a) Ha a barnapártiak pontosan 15%-nyian lennén...         1
31   Egy párt a legutóbbi választáson 36,1%-ot szer...         1
33   Egy párt a legutóbbi választáson 36,1%-ot szer...         1
39   (összesen 25 pont) Egy párt tavaly a választás...         1
40   Tegyük fel, hogy a Bohócpárt támogatottsága az...         1
..                                                 ...       ...
890  2. feladat: Az Egyesült Államok-beli Wind Ener...         1
901  mire kérdez rá, mit kell nézni? Fent a két nem...         1
923  A Cornell University egy kutatója azt vizsgált...         1
931  Egy mobilszolgáltató 3 ügyfélköréből vett egys...         1
970  Szia! Az lenne a kérdésem, hogy ez milyen típu...         1

[125 rows x 2 columns]


Five questions were manually reclassified from 1 to 0 after reviewing the algorithmic matches:

- Row 336, chat ID 26, question ID 44
- Row 351, chat ID 26, question ID 59
- Row 356, chat ID 27, question ID 2
- Row 779, chat ID 91, question ID 1
- Row 862, chat ID 110, question ID 1

After the manual corrections, 120 questions were classified as homework copies.

In [10]:
manual_corrections = [336, 351, 356, 779, 862]

df.loc[
    df['row_id'].isin(manual_corrections),
    'homework'
] = 0

## Homework copying and course relevance

The relationship between homework copying and course relevance was examined using a cross-tabulation.

In [11]:
print(
    pd.crosstab(
        df['homework'],
        df['relevance'],
        margins=True
    )
)

relevance    0    1  All
homework                
0          128  742  870
1            0  120  120
All        128  862  990


The final classification identified 120 homework-copying questions and 128 irrelevant questions. The cross-tabulation showed that 742 questions were relevant course-related questions and were not classified as homework copies. No overlap was observed between irrelevant questions and homework-copying questions.

## User-level homework copying

A student-level measure was created to capture the proportion of a student's questions that were classified as homework copies.

In [12]:
hw_share = (
    df.groupby('id')['homework']
      .mean()
      .reset_index(name='hw_share')
)

df = df.merge(
    hw_share,
    on='id',
    how='left'
)

## Homework copying by reward period

Homework-copying rates were calculated separately for the rewarded and non-rewarded periods for each student.

In [13]:
df['hw_rw'] = df.loc[df['non_rewarded'] == 0, 'homework']
df['hw_nrw'] = df.loc[df['non_rewarded'] == 1, 'homework']

hw_period = (
    df.groupby('id')[['hw_rw', 'hw_nrw']]
      .mean()
      .reset_index()
      .rename(columns={
          'hw_rw': 'avg_hw_rw',
          'hw_nrw': 'avg_hw_nrw'
      })
)

df = df.merge(hw_period, on='id', how='left')

In [14]:
# Move row_id to the first column
cols = ['row_id'] + [col for col in df.columns if col != 'row_id']
df = df[cols]

In [15]:
df.to_excel('databases/chat_level_final.xlsx', index=False)